# Schlieren POD and quantum-kernel SVM — Anaconda / Jupyter

Process your Windows TIFF sequence in notebook cells, starting with 200 consecutive frames. The same processing code produces the saved paper figures and scales to a larger record.

**This notebook contains no experimental results yet.** It uses Mach 2.0, 20 psia absolute stagnation pressure, and 51,032 fps as the supplied acquisition conditions. Clusters are candidate statistical states; the quantum kernel is an exact classical simulation.

## Goal

Choose the image folder and one fixed crop, check a first-frame preview, run the pilot, and inspect POD and classification diagnostics. Keep the original TIFF files unchanged. Camera file order must be complete and uniformly sampled; about 3.9 ms of data in a 200-frame pilot is not enough to establish converged flow statistics.

After a supported run, inspect individual TIFF images grouped by candidate and compare validation-only learning curves from saved modal coefficients. The latter keeps the existing POD representation, training scaler, and candidate assignments fixed while studying classifier sample count and quantum encoding scale.

## Setup

Open this notebook from the extracted **Hemisphere-QML** folder in Jupyter launched through Anaconda Navigator. Run cells with **Shift+Enter**. You can use your current Anaconda environment; a separate environment is optional. Python 3.10 or newer is required.

The next cell prints the exact Python interpreter used by this notebook. If a later import fails, use the optional `%pip` cell and then restart the kernel.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()  # Usually the folder containing this notebook.
# If needed, replace the line above with Path(r"C:\Users\YOUR_NAME\Documents\Hemisphere-QML").
if not (PROJECT_DIR / "schlieren_qml" / "cli.py").is_file():
    raise FileNotFoundError("Set PROJECT_DIR to the extracted Hemisphere-QML folder, then rerun this cell.")
if sys.version_info < (3, 10):
    raise RuntimeError("Choose an Anaconda kernel with Python 3.10 or newer.")
sys.path.insert(0, str(PROJECT_DIR)) if str(PROJECT_DIR) not in sys.path else None
requirements_path = PROJECT_DIR / "requirements.txt"
print("Notebook Python:", sys.executable)
print("Project folder:", PROJECT_DIR)

In [ ]:
# OPTIONAL: uncomment the next line only if an analysis package is missing.
# %pip install -r "{requirements_path}"
# After installing or upgrading packages, restart the kernel and run from the top.

In [ ]:
%matplotlib inline
import json
from copy import deepcopy
from importlib.metadata import version
from packaging.requirements import Requirement
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from schlieren_qml.notebook import (
    resolve_notebook_config, preview_notebook_roi,
    run_from_notebook, display_saved_figures,
)

from schlieren_qml.candidates import candidate_gallery, candidate_image, export_candidate_lists
from schlieren_qml.experiments import validation_learning_curve

package_versions = {}
for line in requirements_path.read_text().splitlines():
    if not line.strip() or line.lstrip().startswith("#"):
        continue
    requirement = Requirement(line)
    actual = version(requirement.name)
    package_versions[requirement.name] = actual
    if actual not in requirement.specifier:
        raise RuntimeError(f"{requirement.name} {actual} is incompatible. Run the optional %pip cell, restart the kernel, then rerun from the top.")
display(package_versions)
print("Notebook helpers are ready.")
from schlieren_qml.pod import load_pod
from schlieren_qml.figures import plot_spatial_modes

## Steps

### 1. Set your folder and crop

Edit **IMAGE_DIRECTORY** below. Use a raw Windows string such as `Path(r"D:\Schlieren\Mach2_run01")`, or forward slashes. Keep outputs in a separate folder outside the image folder.

Start with `ROI = None` to see the full image. Then set `[x0, y0, x1, y1]` in original sensor pixels and rerun the parameters and preview cells. The upper limits are excluded. For a 768×512 image, the full field is `[0, 0, 768, 512]`. A fixed crop is applied to every frame.

The classifier caps below are separate from `PILOT_FRAMES`: they control how many examples train/evaluate each SVM. The same settings reach pilot and full runs. `QUANTUM_ANGLE_SCALES` changes only the quantum encoding, after the shared training-fitted angle mapping; it does not change the candidate labels. Start with the four existing features and three requested candidate groups.

In [ ]:
pilot_manifest = None
pilot_run_config = None
INPUTS_READY = False

IMAGE_DIRECTORY = Path(r"D:\Schlieren\Mach2_run01")  # CHANGE THIS.
ROI = None  # Example only: [100, 40, 700, 470]. Choose from your actual image.
START_FRAME = 0  # Index in the naturally sorted file list; skip tunnel startup if necessary.
PILOT_FRAMES = 200
BIN_FACTOR = 1  # Spatial area averaging; 1 retains the original pixels.
FIGURE_DPI = 600
MODE_COLOR_VMAX = 0.008  # Shared [-vmax, +vmax] limits for the original mode figure in all runs.
MODE_COLOR_PERCENTILE = None  # Or 99.5; leave VMAX=None when using a percentile.
PILOT_OUTPUT_DIRECTORY = PROJECT_DIR / "results" / "notebook_pilot"
OVERWRITE_PILOT = False  # Change only when deliberately replacing generated pilot results.

# The full run stays disabled during Run All until you explicitly enable it.
RUN_FULL = False
FULL_BIN_FACTOR = 2
FULL_OUTPUT_DIRECTORY = PROJECT_DIR / "results" / "notebook_full"
OVERWRITE_FULL = False

# Shared classifier controls for both new pilot and full-record runs.
CLASSIFIER_MAX_TRAIN = 512
CLASSIFIER_MAX_VALIDATION = 512
CLASSIFIER_MAX_TEST = 256
CLASSIFIER_C_VALUES = [0.01, 0.1, 1.0, 10.0, 100.0]
QUANTUM_ANGLE_SCALES = [0.1, 0.25, 0.5, 1.0]
QUANTUM_REPETITIONS = [1, 2]
classifier_options = dict(
    max_train=CLASSIFIER_MAX_TRAIN, max_validation=CLASSIFIER_MAX_VALIDATION,
    max_test=CLASSIFIER_MAX_TEST, C_values=CLASSIFIER_C_VALUES,
    angle_scales=QUANTUM_ANGLE_SCALES, reps=QUANTUM_REPETITIONS,
)

# Inspect a saved run independently of whether a new run is enabled above.
REVIEW_RESULTS_DIRECTORY = PROJECT_DIR / "results" / "notebook_full"
IMAGE_DIRECTORY_OVERRIDE = None  # Example: Path(r"D:\Relocated_TIFFs") if source images moved.
GALLERY_EXAMPLES_PER_CANDIDATE = 4
GALLERY_SEED = 42  # Change this to inspect other examples.
GALLERY_LABEL_COLUMN = "state"  # Or "rbf_test_prediction" / "qsvm_test_prediction".
GALLERY_PARTITION = "train"  # For predictions use "test"; None includes all partitions.
GALLERY_PROCESSED = True  # Saved crop/binning; False displays original TIFF pixels.
GALLERY_ONLY_CANDIDATE = None  # Or "candidate_1", "candidate_2", "candidate_3".
FILENAME_TO_VIEW = None  # Enter an exact filename from state_assignments.csv.
EXPORT_CANDIDATE_LISTS = False
CANDIDATE_LIST_OUTPUT_DIRECTORY = PROJECT_DIR / "results" / "candidate_filename_lists"

# Classification-only study from saved coefficients: no TIFF reads or POD refit.
RUN_VALIDATION_STUDY = False
LEARNING_CURVE_TRAIN_SIZES = [256, 512, 1024]
VALIDATION_STUDY_OUTPUT_DIRECTORY = PROJECT_DIR / "results" / "validation_study"
OVERWRITE_VALIDATION_STUDY = False

pilot_config = json.loads((PROJECT_DIR / "configs" / "pilot.json").read_text())
pilot_config.update(
    image_directory=str(IMAGE_DIRECTORY), output_directory=str(PILOT_OUTPUT_DIRECTORY),
    roi=ROI, start=START_FRAME, limit=PILOT_FRAMES, bin_factor=BIN_FACTOR,
)
pilot_config["classifier"].update(deepcopy(classifier_options))
pilot_config["figures"].update(dpi=FIGURE_DPI, mode_color_vmax=MODE_COLOR_VMAX,
                               mode_color_percentile=MODE_COLOR_PERCENTILE)
pilot_config = resolve_notebook_config(pilot_config, PROJECT_DIR)
print("Image folder:", pilot_config["image_directory"])
print("Pilot output:", pilot_config["output_directory"])

### 2. Preview the crop and processing size

Inspect the first frame, check shock/hemisphere coverage, and verify the first/last filenames. The crop preview does not normalize individual frames. The cache estimate is disk space for the float32 image matrix; extra space is needed for other outputs.

In [ ]:
INPUTS_READY = Path(pilot_config["image_directory"]).is_dir()
if INPUTS_READY:
    preview_figure, preview_summary = preview_notebook_roi(pilot_config)
    display(preview_figure)
    plt.close(preview_figure)
    display(preview_summary)
else:
    print("Set IMAGE_DIRECTORY to your TIFF folder, then rerun the parameters and preview cells.")

### 3. Run the 200-frame pilot

The processing runs with the **same Python interpreter as this kernel**, in a separate process. Progress appears below. This preserves inline plotting and releases large file handles when the process ends. You can interrupt this cell to stop the child process.

A second run protects existing results. Choose a new output folder or set `OVERWRITE_PILOT = True` in the parameters cell when you deliberately want to replace them.

In [ ]:
pilot_manifest = None
if INPUTS_READY:
    pilot_run_config = deepcopy(pilot_config)
    pilot_manifest = run_from_notebook(
        pilot_run_config, PROJECT_DIR, overwrite=OVERWRITE_PILOT
    )
    display({key: pilot_manifest.get(key) for key in (
        "status", "classification_status", "frames", "pod_rank",
        "first_to_last_span_s", "pod_retained_training_variance_fraction",
    )})
else:
    print("Pilot not run: set the TIFF folder first.")

### 4. Inspect the saved figures inline

The pipeline exports PDF, SVG, and PNG files under the output folder. The PNGs below are embedded in the notebook display; use PDF/SVG for journal layout. Figures describe uncalibrated schlieren image intensity unless you supply a spatial calibration in the configuration. POD variance is not kinetic energy.

In [ ]:
# Generate the original spatial-mode figure using the parameter-cell limits.
figure_manifest = pilot_manifest
figure_results = Path(pilot_run_config["output_directory"]) if pilot_manifest is not None else REVIEW_RESULTS_DIRECTORY
if figure_manifest is None and (figure_results / "run_manifest.json").is_file():
    figure_manifest = json.loads((figure_results / "run_manifest.json").read_text())
if figure_manifest is not None:
    saved_config = json.loads((figure_results / "config_used.json").read_text())
    frame_metadata = json.loads((figure_results / "frames.json").read_text())
    mode_options = dict(saved_config.get("figures", {}))
    mode_options.update(mode_color_vmax=MODE_COLOR_VMAX,
                        mode_color_percentile=MODE_COLOR_PERCENTILE,
                        n_modes_plot=4, dpi=FIGURE_DPI,
                        bin_factor=frame_metadata.get("bin_factor", 1))
    plot_spatial_modes(load_pod(figure_results / "pod_model.npz"),
                       tuple(frame_metadata["image_shape"]),
                       figure_results / "figures", config=mode_options)
    notes_path = figure_results / "figures" / "FIGURE_NOTES.md"
    if notes_path.is_file():
        mode_note = (figure_results / "figures" / "04_spatial_modes_notes.md").read_text().strip()
        lines = notes_path.read_text().splitlines()
        lines = [mode_note if line.startswith("04:") else line for line in lines]
        if not any(line.startswith("04:") for line in lines):
            lines.append(mode_note)
        notes_path.write_text("\n".join(lines) + "\n")
    figure_count = display_saved_figures(figure_manifest, width=950)
    print(f"Displayed {figure_count} figures. Original mode figure: {figure_results / 'figures' / '04_spatial_modes.png'}")
    mode_scale = json.loads((figure_results / "figures" / "04_spatial_modes_color_scale.json").read_text())
    print("Applied mode color limits:", mode_scale["vmin"], mode_scale["vmax"])
else:
    print("Run the pilot or set REVIEW_RESULTS_DIRECTORY to your completed results folder.")

### 5. Check class coverage, then read classifier scores

The coverage table comes first because a validation or test block can omit candidate states. A single-class test block measures that state's recall, not full multiclass performance. Missing states are flagged even when accuracy is high. The reported balanced accuracy averages classes present in the evaluated block; macro-F1 includes every declared class and assigns zero to absent classes.

These targets are training-defined K-means assignments because independent physical labels are unavailable. The scores assess reproduction of that assignment rule. Inspect class support, temporal correlation, clipping and warnings alongside accuracy.

In [ ]:
if pilot_manifest is not None:
    results_folder = Path(pilot_run_config["output_directory"])
    metrics_path = results_folder / "classification_metrics.json"
    if metrics_path.is_file():
        classification_metrics = json.loads(metrics_path.read_text())
        intended_classes = classification_metrics["classes"]
        sampled_counts = classification_metrics.get("sampled_class_counts", {})
        coverage_table = {
            partition: {state: counts.get(state, 0) for state in intended_classes}
            for partition, counts in sampled_counts.items()
        }
        display({"evaluated_frame_counts_by_partition": coverage_table})
        for partition in ("validation", "test"):
            if partition not in coverage_table:
                print(f"CLASS COVERAGE WARNING: {partition} class counts are unavailable.")
                continue
            observed = [state for state, count in coverage_table[partition].items() if count > 0]
            missing = [state for state in intended_classes if state not in observed]
            if missing:
                print(f"CLASS COVERAGE WARNING: {partition} covers {len(observed)} of {len(intended_classes)} intended states; missing {missing}. Scores do not assess all intended states.")
        display({model: {key: values[key] for key in (
            "accuracy", "balanced_accuracy", "macro_f1", "n", "class_support"
        )} for model, values in classification_metrics["test"].items()})
        display({"label_mode": classification_metrics["label_mode"],
                 "warnings": classification_metrics["warnings"]})
    else:
        print("Classification was disabled or unavailable. Inspect the POD outputs and run manifest.")
else:
    print("Classifier scores will appear after a supported pilot run.")

### 6. Make an editable notebook plot

This cell loads only the compact modal coefficients, not the large image cache. Edit this plotting code to explore the time histories. Avoid opening `image_matrix.npy` with memory mapping in this kernel while rerunning processing on Windows.

In [ ]:
if pilot_manifest is not None:
    coefficients = np.load(Path(pilot_run_config["output_directory"]) / "modal_coefficients.npy", allow_pickle=False)
    time_ms = np.arange(len(coefficients)) * pilot_run_config.get("stride", 1) / pilot_run_config["fps"] * 1000
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(time_ms, coefficients[:, 0], color="#0072B2", linewidth=1)
    ax.set(xlabel="Time from first selected frame (ms)",
           ylabel="Mode 1 coefficient (a.u.)", title="POD coefficient from the frozen training basis")
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    print("Coefficient array shape:", coefficients.shape)

## Checks

Before interpreting results, verify file order and dropped frames, crop coverage, illumination/camera stability, and the short record length. Inspect training autocorrelation before choosing temporal gaps, and assess POD/state stability with longer records and repeat runs. The initial settings are exploratory.

### Optional: run the larger record

Only after reviewing the pilot, set `RUN_FULL = True`, rerun the **parameters cell**, rerun the **preview cell**, and then run the full-record cell below. You do not need to rerun the pilot cell into its existing output folder. It uses the full configuration's POD rank and temporal gap with your folder/crop/start, and defaults to two-pixel spatial binning. Crop width and height must divide by the bin factor. A 20,000-frame float32 cache at 768×512 requires about 31.46 GB unbinned or 7.86 GB with 2× binning, before other outputs.

This execution also uses the shared classifier caps, C grid, angle-scale grid, and repetition settings from the parameters cell. The complete record still supports POD and temporal diagnostics; only the SVM examples are capped. To explore classifier settings without reprocessing the TIFFs, use the saved-results study below instead.

In [ ]:
full_manifest = None
if RUN_FULL and INPUTS_READY:
    full_config = json.loads((PROJECT_DIR / "configs" / "full.json").read_text())
    full_config.update(
        image_directory=pilot_config["image_directory"],
        output_directory=str(FULL_OUTPUT_DIRECTORY), roi=ROI,
        start=START_FRAME, bin_factor=FULL_BIN_FACTOR,
    )
    full_config["classifier"].update(deepcopy(classifier_options))
    full_config["figures"].update(dpi=FIGURE_DPI, mode_color_vmax=MODE_COLOR_VMAX,
                                  mode_color_percentile=MODE_COLOR_PERCENTILE)
    full_config = resolve_notebook_config(full_config, PROJECT_DIR)
    full_manifest = run_from_notebook(full_config, PROJECT_DIR, overwrite=OVERWRITE_FULL)
    display({key: full_manifest.get(key) for key in ("status", "frames", "elapsed_s")})
else:
    print("Full run disabled. Review the pilot before setting RUN_FULL = True.")

### Inspect images in each candidate group

Set `REVIEW_RESULTS_DIRECTORY` in the parameters cell to the completed run you want to inspect. The saved `state_assignments.csv` maps every processed filename to its nearest-center candidate; SVM prediction columns are filled only for the capped test examples. Blank predictions elsewhere are expected.

The gallery uses the saved crop and binning, not newly edited image-processing settings. It defaults to **training images** so physical interpretation can be developed before inspecting a fresh holdout. Change `GALLERY_SEED` to see different examples, or `GALLERY_ONLY_CANDIDATE` to inspect one group. For a prediction gallery, set `GALLERY_LABEL_COLUMN` to an SVM prediction column and `GALLERY_PARTITION = "test"` (or `None`). Each panel retains its assigned label and available prediction.

All panels share grayscale limits. Set `GALLERY_PROCESSED = False` to inspect the original TIFF pixels. If TIFFs were moved, set `IMAGE_DIRECTORY_OVERRIDE` to their new folder.

In [ ]:
if (REVIEW_RESULTS_DIRECTORY / "state_assignments.csv").is_file():
    candidate_examples = candidate_gallery(
        REVIEW_RESULTS_DIRECTORY,
        examples_per_candidate=GALLERY_EXAMPLES_PER_CANDIDATE,
        seed=GALLERY_SEED, label_column=GALLERY_LABEL_COLUMN,
        partition=GALLERY_PARTITION, processed=GALLERY_PROCESSED,
        image_directory_override=IMAGE_DIRECTORY_OVERRIDE,
        only_candidate=GALLERY_ONLY_CANDIDATE,
    )
    display(candidate_examples["figure"])
    plt.close(candidate_examples["figure"])
    display({"matching_candidate_counts": candidate_examples["counts"],
             "label_column": candidate_examples["label_column"],
             "partition": candidate_examples["partition"]})
    display([{key: row.get(key) for key in ("filename", "state", "partition", "time_s",
                                            "rbf_test_prediction", "qsvm_test_prediction")}
             for row in candidate_examples["selected_rows"]])
else:
    print("Set REVIEW_RESULTS_DIRECTORY to a completed run with state_assignments.csv.")

### Inspect one filename or export complete candidate lists

Set `FILENAME_TO_VIEW` in the parameters cell to show one image and its assignment. To identify every image in a candidate, open `state_assignments.csv` in Excel and filter its `state` column, or enable the optional CSV export below. The export creates filename lists and does not duplicate or modify the TIFFs. Use a fresh list-output folder to preserve previous exports.

In [ ]:
if FILENAME_TO_VIEW is not None:
    selected_image = candidate_image(
        REVIEW_RESULTS_DIRECTORY, FILENAME_TO_VIEW,
        processed=GALLERY_PROCESSED,
        image_directory_override=IMAGE_DIRECTORY_OVERRIDE,
    )
    display(selected_image["figure"])
    plt.close(selected_image["figure"])
    display(selected_image["row"])
else:
    print("Set FILENAME_TO_VIEW to inspect a particular TIFF.")

if EXPORT_CANDIDATE_LISTS:
    candidate_exports = export_candidate_lists(
        REVIEW_RESULTS_DIRECTORY, CANDIDATE_LIST_OUTPUT_DIRECTORY,
    )
    display(candidate_exports)
else:
    print("Candidate list export is disabled; state_assignments.csv already contains all assignments.")

### Validation-only classifier learning curves

This optional study reads saved coefficients and assignments from `REVIEW_RESULTS_DIRECTORY`; it does **not** reread TIFFs, refit POD, rediscover clusters, or evaluate the test block. Existing labels, the POD feature definition, and training-fitted scaling remain fixed while classifier training caps and quantum encoding scale are compared. Both classifiers use the same examples at each training size, and the validation examples stay fixed across the curve.

Set `RUN_VALIDATION_STUDY = True` and choose a fresh `VALIDATION_STUDY_OUTPUT_DIRECTORY`, then rerun parameters and this study cell. The default compares 256, 512, and 1,024 training examples, capped by what is actually available, with up to 512 validation examples. Actual sample counts and class coverage are recorded. Kernel storage/work grows approximately with the square of the classifier training count.

Use **validation balanced accuracy and macro-F1** to guide changes. More training, stronger C, more repetitions, or different encoding scales do not guarantee improvement. A training-to-validation gap should be assessed alongside the learning curve. The quantum calculation is an exact classical statevector simulation.

The previously viewed test matrix must not guide these choices. After selecting the model settings, reserve a fresh chronological block or independent tunnel run for the final comparison and assess uncertainty using justified temporal blocks/runs.

In [ ]:
validation_study = None
if RUN_VALIDATION_STUDY:
    validation_study = validation_learning_curve(
        REVIEW_RESULTS_DIRECTORY, VALIDATION_STUDY_OUTPUT_DIRECTORY,
        train_sizes=LEARNING_CURVE_TRAIN_SIZES,
        max_validation=CLASSIFIER_MAX_VALIDATION,
        angle_scales=QUANTUM_ANGLE_SCALES,
        C_values=CLASSIFIER_C_VALUES, reps=QUANTUM_REPETITIONS,
        dpi=FIGURE_DPI, overwrite=OVERWRITE_VALIDATION_STUDY,
    )
    display_saved_figures(validation_study, width=1000)
    study_rows = [
        {"training_examples": run["n_train"], "validation_examples": run["n_validation"],
         "model": model,
         "training_balanced_accuracy_pct": round(100*run["training"][model]["balanced_accuracy"], 2),
         "validation_balanced_accuracy_pct": round(100*run["validation"][model]["balanced_accuracy"], 2),
         "validation_macro_f1_pct": round(100*run["validation"][model]["macro_f1"], 2),
         "selected_parameters": run["selected_hyperparameters"].get(model, {})}
        for run in validation_study["runs"]
        for model in ("rbf_svm", "qsvm", "majority")
    ]
    display(study_rows)
    display({"warnings": validation_study.get("warnings", [])})
    print("Validation study saved to:", validation_study["output_directory"])
else:
    print("Validation-only study disabled. Set RUN_VALIDATION_STUDY = True to compare saved-run classifier settings.")

## Next steps

Compare the candidate gallery with `11_state_conditional_means.png`, several frames per group, and the spatial POD modes. Three requested clusters do not establish three distinct physical flow regimes. Keep candidate labels fixed during classifier comparisons; changing crop, features, or cluster count changes the task unless you explicitly preserve an independent target definition.

Use the validation-only study to investigate sample-size and encoding sensitivity. Preserve a fresh final holdout; already viewed test scores are exploratory. Record hemisphere diameter, exposure time, spatial calibration, POD/state convergence, and repeat-run information before preparing paper claims.

See `docs/ANACONDA_JUPYTER.md` for setup and saved-run instructions and `docs/PAPER_GUIDE.md` for methodology. No experimental result or quantum advantage is asserted by this unexecuted starter notebook.